# M2 Circuit Selection — Hur et al. Front-End Experiment

**Purpose:** Determine the best Hur et al. (2022) ansatz for the trainable quanvolutional front-end of M2.

Three circuits are tested as drop-in replacements for Long et al.'s fixed `RandomLayers` front-end:

| Circuit | Structure | Role | Params (shared) | CNOTs |
|---------|-----------|------|----------------|-------|
| **Circuit 2** | Ry–Ry–CNOT–Ry–Ry–CNOT | Ablation control (high entanglement, low axis diversity) | 4 | 2 |
| **Circuit 4** | Ry–Rz–CNOT–Ry–Rz–CNOT(rev)–Ry–Rz | Primary candidate (balanced) | 6 | 2 |
| **Circuit 6** | SO(4) gate — arbitrary real 2-qubit unitary | Theoretical motivator | 6 | 3 |

All other components (encoding, classical backbone, quantum back-end) are identical to M1.  
Weight sharing is applied translationally across all qubit pairs — circuits are applied to pairs (0,1), (1,2), (2,3), (3,0) with **shared weights**.

**Decision criterion:** The circuit with highest MNIST 3v5 validation accuracy at noiseless 0% IBM noise becomes the locked M2 front-end.  
If two circuits tie, prefer the shallower one (fewer CNOTs) for noise robustness.

## 1. Environment & Configuration

In [ ]:
# ── Gadi conda env: qcqcnn  ──────────────────────────────────────────────────
# Run from:  /g/data/wa66/yh2481/
# PBS queue: gpuvolta   (batch size 1 mandatory for quantum layers)
# Note: GPU utilisation will be 0% — quantum layers run on CPU via Qiskit-Aer
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, time, json, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml
from pennylane.templates import RandomLayers

import tensorflow as tf
from tensorflow import keras

from qiskit import QuantumCircuit
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

warnings.filterwarnings('ignore')
print(f'PyTorch   : {torch.__version__}')
print(f'PennyLane : {qml.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device    : {device}')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Smoke test first: set SMOKE_TEST = True for quick gradient/loss check
# Then flip to False for the full circuit selection run
SMOKE_TEST = False

CFG = {
    # Data
    'target_labels'    : [3, 5],
    'per_class_train'  : 50  if SMOKE_TEST else 500,   # samples per class
    'per_class_val'    : 25  if SMOKE_TEST else 100,
    'per_class_test'   : 50  if SMOKE_TEST else 200,

    # Quantum front-end (fixed, matches Long et al.)
    'n_qubits'         : 4,
    'filter_size'      : 2,
    'stride'           : 2,

    # Training — batch size 1 MANDATORY for quantum layers
    'batch_size'       : 1,
    'lr'               : 1e-3,
    'epochs'           : 10 if SMOKE_TEST else 30,
    'patience'         : 5  if SMOKE_TEST else 10,
    'seed'             : 42,

    # Output
    'results_dir'      : 'results_circuit_selection',
    'cache_dir'        : 'cache_circuit_selection',
}

os.makedirs(CFG['results_dir'], exist_ok=True)
os.makedirs(CFG['cache_dir'], exist_ok=True)

# Reproducibility
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

MODE = 'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'
print(f'Mode: {MODE}')
print(f"Train: {CFG['per_class_train']*2} | Val: {CFG['per_class_val']*2} | Test: {CFG['per_class_test']*2}")
print(f"Epochs: {CFG['epochs']} | LR: {CFG['lr']} | Batch: {CFG['batch_size']}")

## 2. Data Loading & Quanvolutional Pre-processing

The quantum front-end is applied **once** to produce a cached 14×14×4 feature map per image.  
Each circuit variant gets its **own fresh pre-processing pass** with freshly initialised trainable weights — this is the key difference from M1 (fixed random weights).

> **Important:** In M1, `RandomLayers` weights are fixed at init time and never updated.  
> In M2, the circuit weights start random but are updated via the parameter-shift rule during training.

In [ ]:
# ── Load MNIST 3v5 ────────────────────────────────────────────────────────────
(x_tr_raw, y_tr_raw), (x_te_raw, y_te_raw) = keras.datasets.mnist.load_data()

def filter_and_sample(images, labels, target_labels, per_class, seed=42):
    rng = np.random.default_rng(seed)
    selected_imgs, selected_lbls = [], []
    for cls in target_labels:
        idx = np.where(labels == cls)[0]
        chosen = rng.choice(idx, size=per_class, replace=False)
        selected_imgs.append(images[chosen])
        # Remap: first target_label → 0, second → 1
        selected_lbls.append(np.full(per_class, target_labels.index(cls)))
    imgs = np.concatenate(selected_imgs).astype(np.float32) / 255.0
    lbls = np.concatenate(selected_lbls).astype(np.float32)
    # Shuffle
    perm = rng.permutation(len(imgs))
    return imgs[perm], lbls[perm]

# Training + validation split from training set
n_per_class_total = CFG['per_class_train'] + CFG['per_class_val']
x_all, y_all = filter_and_sample(
    x_tr_raw, y_tr_raw, CFG['target_labels'], n_per_class_total, seed=CFG['seed']
)
n_total   = len(x_all)
n_train   = CFG['per_class_train'] * 2
x_train, y_train = x_all[:n_train],  y_all[:n_train]
x_val,   y_val   = x_all[n_train:],  y_all[n_train:]

x_test, y_test = filter_and_sample(
    x_te_raw, y_te_raw, CFG['target_labels'], CFG['per_class_test'], seed=CFG['seed']+1
)

# Add channel dim: (N, 28, 28) → (N, 28, 28, 1)
x_train = x_train[..., np.newaxis]
x_val   = x_val[..., np.newaxis]
x_test  = x_test[..., np.newaxis]

print(f'Train : {x_train.shape}  labels: {np.unique(y_train)}')
print(f'Val   : {x_val.shape}')
print(f'Test  : {x_test.shape}')

## 3. The Three Hur et al. Ansatz Circuits

Each function implements a **2-qubit trainable block** applied translationally invariant across qubit pairs (0,1), (1,2), (2,3), (3,0) with **shared weights**.

The encoding (Ry angle encoding) is identical for all three — only the parameterised unitary after encoding changes.

In [ ]:
# ── Circuit 2 — high entanglement, Ry-only rotations ─────────────────────────
# Origin: Sim et al. (2019) circuit with best entangling capability → 2-qubit form by Hur et al.
# Structure: Ry–Ry–CNOT–Ry–Ry–CNOT
# Parameters: 4 shared (θ0=Ry q0 pre, θ1=Ry q1 pre, θ2=Ry q0 post, θ3=Ry q1 post)
# CNOTs: 2  |  IBM-native: yes  |  Role: ablation control

def circuit2_block(weights, wire0, wire1):
    """Hur et al. Circuit 2 — 4 params, 2 CNOTs, Ry-only."""
    qml.RY(weights[0], wires=wire0)
    qml.RY(weights[1], wires=wire1)
    qml.CNOT(wires=[wire0, wire1])
    qml.RY(weights[2], wires=wire0)
    qml.RY(weights[3], wires=wire1)
    qml.CNOT(wires=[wire1, wire0])   # reversed direction


# ── Circuit 4 — balanced expressibility + entanglement (PRIMARY CANDIDATE) ───
# Origin: Sim et al. (2019) balanced circuits → 2-qubit form by Hur et al.
# Structure: Ry–Rz–CNOT–Ry–Rz–CNOT(rev)–Ry–Rz
# Parameters: 6 shared (θ0=Ry q0, θ1=Ry q1, θ2=Rz q0, θ3=Rz q1,
#                       θ4=Ry q0 post, θ5=Rz q0 post  — see gate order below)
# CNOTs: 2  |  IBM-native: yes  |  Role: primary M2 candidate
# Note: Rz gates add phase diversity not present in Circuit 2 — this is the
# key structural difference that gives C4 greater expressibility.

def circuit4_block(weights, wire0, wire1):
    """Hur et al. Circuit 4 — 6 params, 2 CNOTs, Ry+Rz axis diversity."""
    # Pre-entanglement rotations
    qml.RY(weights[0], wires=wire0)
    qml.RZ(weights[1], wires=wire0)
    qml.RY(weights[2], wires=wire1)
    # First entangling CNOT
    qml.CNOT(wires=[wire0, wire1])
    # Post-entanglement rotations
    qml.RY(weights[3], wires=wire0)
    qml.RZ(weights[4], wires=wire0)
    qml.RY(weights[5], wires=wire1)
    # Second entangling CNOT — reversed direction
    qml.CNOT(wires=[wire1, wire0])


# ── Circuit 6 — SO(4) gate, maximal real unitary ──────────────────────────────
# Origin: VQE two-body entangler literature → adopted by Hur et al. for QCNN conv layer
# This circuit can implement ANY real 2-qubit unitary (SO(4) group coverage)
# Structure: Ry–CNOT–Ry–Rz–CNOT–Ry–Rz–CNOT–Ry  (approximation of canonical SO(4) decomp)
# Parameters: 6 shared  |  CNOTs: 3  |  IBM-native: yes (but marginal noise tolerance)
# Role: theoretical upper bound — if C6 >> C4, real-unitary expressibility matters

def circuit6_block(weights, wire0, wire1):
    """Hur et al. Circuit 6 — SO(4) gate, 6 params, 3 CNOTs."""
    # Layer 1
    qml.RY(weights[0], wires=wire0)
    qml.CNOT(wires=[wire0, wire1])
    # Layer 2
    qml.RY(weights[1], wires=wire0)
    qml.RZ(weights[2], wires=wire0)
    qml.CNOT(wires=[wire1, wire0])
    # Layer 3
    qml.RY(weights[3], wires=wire0)
    qml.RZ(weights[4], wires=wire0)
    qml.CNOT(wires=[wire0, wire1])
    # Final rotation
    qml.RY(weights[5], wires=wire1)


# Parameter counts for reference
CIRCUIT_SPECS = {
    'circuit2': {'n_params': 4, 'n_cnots': 2, 'block_fn': circuit2_block},
    'circuit4': {'n_params': 6, 'n_cnots': 2, 'block_fn': circuit4_block},
    'circuit6': {'n_params': 6, 'n_cnots': 3, 'block_fn': circuit6_block},
}
for name, spec in CIRCUIT_SPECS.items():
    print(f'{name}: {spec["n_params"]} trainable params | {spec["n_cnots"]} CNOTs')

## 4. Trainable Quanvolutional Layer

This is the **core M2 contribution** — a `torch.nn.Module` wrapping a PennyLane `TorchLayer` that replaces `RandomLayers`.  
The key design choices:
- **Weight sharing** across all 4 qubit pairs — same weights applied to (0,1), (1,2), (2,3), (3,0)
- **Angle encoding** via `RY(π × pixel)` — identical to M1
- **Parameter-shift rule** gradients via PennyLane `TorchLayer`
- **Output:** 14×14×4 feature map — identical shape to M1 output

In [ ]:
# ── Trainable quanvolutional module ──────────────────────────────────────────

def make_trainable_quanv_layer(circuit_name, n_qubits=4):
    """
    Build a trainable quanvolutional TorchLayer for a given Hur et al. circuit.
    
    Returns:
        TorchLayer with 'weights' as the trainable parameter key.
        The layer maps a (4,) pixel patch input to (4,) expectation values.
    """
    spec      = CIRCUIT_SPECS[circuit_name]
    block_fn  = spec['block_fn']
    n_params  = spec['n_params']
    dev       = qml.device('default.qubit', wires=n_qubits)

    # Qubit pairs for translationally-invariant application
    qubit_pairs = [(i, (i+1) % n_qubits) for i in range(n_qubits)]

    @qml.qnode(dev, interface='torch', diff_method='parameter-shift')
    def quantum_circuit(inputs, weights):
        # Angle encoding: RY(pi * pixel) on each qubit
        # inputs shape: (n_qubits,) = one 2×2 patch flattened
        for i in range(n_qubits):
            qml.RY(np.pi * inputs[i], wires=i)

        # Translationally invariant application — shared weights across all pairs
        for wire0, wire1 in qubit_pairs:
            block_fn(weights, wire0, wire1)

        # Measure all qubits in the Z basis
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    # Wrap as a TorchLayer — 'weights' key registers the parameter tensor for autograd
    weight_shapes = {'weights': (n_params,)}
    torch_layer   = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

    return torch_layer, n_params


class TrainableQuanvLayer(nn.Module):
    """
    Applies the trainable quantum circuit as a sliding 2×2 window over the input image.
    
    Input  : (N, H, W, 1) numpy array  [N images, 28×28 grayscale]
    Output : (N, H//stride, W//stride, n_qubits) tensor  [14×14×4]
    
    This is the M2 replacement for Long et al.'s fixed RandomLayers quanvolutional layer.
    """
    def __init__(self, circuit_name, filter_size=2, stride=2, n_qubits=4):
        super().__init__()
        self.filter_size  = filter_size
        self.stride       = stride
        self.n_qubits     = n_qubits
        self.circuit_name = circuit_name

        self.torch_layer, self.n_params = make_trainable_quanv_layer(
            circuit_name, n_qubits
        )
        print(f'  [{circuit_name}] TrainableQuanvLayer: {self.n_params} trainable params')

    def forward(self, images):
        """
        images: (N, H, W, 1) torch tensor
        returns: (N, H//stride, W//stride, n_qubits) torch tensor
        """
        N, H, W, _ = images.shape
        out_H = (H - self.filter_size) // self.stride + 1
        out_W = (W - self.filter_size) // self.stride + 1

        out = torch.zeros(N, out_H, out_W, self.n_qubits)

        for i in range(0, H - self.filter_size + 1, self.stride):
            for j in range(0, W - self.filter_size + 1, self.stride):
                # Extract 2×2 patch and flatten to (N, 4)
                patch = images[:, i:i+self.filter_size, j:j+self.filter_size, 0]
                patch = patch.reshape(N, self.filter_size * self.filter_size)

                # Process each sample individually (batch_size=1 semantics for QNN)
                for n in range(N):
                    q_out = self.torch_layer(patch[n])  # (n_qubits,)
                    out[n, i//self.stride, j//self.stride] = q_out

        return out


print('TrainableQuanvLayer defined.')
print('Note: forward pass loops over patches — this is the expected bottleneck.')
print('GPU utilisation will be 0% (quantum layers run on CPU via PennyLane).')

In [ ]:
# ── Pre-compute feature maps ──────────────────────────────────────────────────
# Each circuit gets its own quanvolutional pass with freshly-init weights.
# We DON'T cache between circuits — each must start from its own random init.
#
# NOTE: For the smoke test / circuit selection experiment, we pre-process ONCE
# per circuit at init, then the classical backbone trains on the fixed feature map.
# This is valid for circuit selection purposes (testing which circuit learns best
# when its parameters are frozen after init is NOT what we want — see the full
# M2 model below where the front-end weights are actually trained end-to-end).
#
# For the FULL M2 model, the quanvolutional layer recomputes during training.

def precompute_feature_maps(images, layer, desc=''):
    """
    Pre-process images through quanvolutional layer.
    Returns (N, 14, 14, 4) feature maps as numpy array.
    """
    imgs_tensor = torch.tensor(images, dtype=torch.float32)
    print(f'  Pre-computing {desc} features for {len(images)} images...')
    t0 = time.time()
    with torch.no_grad():
        feat = layer(imgs_tensor)
    elapsed = time.time() - t0
    print(f'  Done: {feat.shape}  ({elapsed:.1f}s)')
    return feat.numpy()

print('Pre-compute function ready. Feature maps generated per circuit in Section 6.')

## 5. Classical Backbone & Quantum Back-End

Identical to M1 — the backbone is **not changed** between circuits.  
This isolates front-end performance as the only variable.

In [ ]:
# ── Quantum back-end (Long et al. QNN classifier) ─────────────────────────────
# 2 qubits, ZZFeatureMap(2) + RealAmplitudes(2, reps=1) → 4 trainable params
# This is unchanged from M1 (the Long et al. replication)

def build_qnn_backend():
    """Build the Qiskit-based QNN classifier used in Long et al."""
    feature_map = ZZFeatureMap(feature_dimension=2, reps=1)
    ansatz      = RealAmplitudes(num_qubits=2, reps=1)
    qc          = QuantumCircuit(2)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz,      inplace=True)

    qnn = EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
    )
    return TorchConnector(qnn)


# ── Full M2 model (trainable front-end + classical CNN + QNN back-end) ─────────

class M2_Net(nn.Module):
    """
    Full M2 model — trainable QCQ-CNN.
    
    Architecture:
      [Trainable Quanv (Hur et al. circuit)] → [Classical CNN backbone] → [QNN classifier]
    
    Input:  (N, 28, 28, 1) image tensor
    Output: (N, 1) logit
    
    Parameters:
      - Quantum front-end: circuit2=4, circuit4=6, circuit6=6 trainable params
      - Classical CNN backbone: ~1,590 params (identical to M1)
      - QNN back-end: 4 params (identical to M1)
    """
    def __init__(self, circuit_name):
        super().__init__()
        self.circuit_name = circuit_name

        # ── Trainable quanvolutional front-end (M2 contribution) ─────────────
        self.quanv = TrainableQuanvLayer(circuit_name)

        # ── Classical CNN backbone (identical to M1/Long et al.) ─────────────
        # Input:  14×14×4 feature map  (channels-first after permute)
        self.conv1  = nn.Conv2d(4, 8, kernel_size=3)    # → 12×12×8
        self.pool1  = nn.MaxPool2d(2)                   # → 6×6×8
        self.conv2  = nn.Conv2d(8, 16, kernel_size=3)   # → 4×4×16
        self.pool2  = nn.MaxPool2d(2)                   # → 2×2×16 = 64
        self.fc2    = nn.LazyLinear(2)                  # 64 → 2  (lazy: infers 64)

        # ── QNN back-end (identical to M1/Long et al.) ───────────────────────
        self.qnn = build_qnn_backend()
        self.fc3 = nn.Linear(1, 2)

    def forward(self, x):
        # x: (N, 28, 28, 1)

        # 1. Trainable quantum front-end → (N, 14, 14, 4)
        x = self.quanv(x)

        # 2. Permute to channels-first for PyTorch Conv2d: (N, 4, 14, 14)
        x = x.permute(0, 3, 1, 2).float()

        # 3. Classical CNN backbone
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = self.fc2(x)               # → (N, 2)

        # 4. QNN back-end (takes 2 features as input)
        x = self.qnn(x)               # → (N, 1)

        # 5. Final linear
        x = self.fc3(x)               # → (N, 2)

        return x


print('M2_Net defined.')

In [ ]:
# ── Parameter count verification ─────────────────────────────────────────────
def count_parameters(model, label=''):
    """
    Count and report trainable parameters separately for
    quantum front-end, classical backbone, and quantum back-end.
    
    Note: Quantum and classical params are reported separately because
    they represent fundamentally different computational resources:
      - Quantum params updated via parameter-shift rule (2 circuit evals per param per step)
      - Classical params updated via standard SGD/Adam backprop
    """
    quanv_params     = sum(p.numel() for p in model.quanv.parameters())
    classical_params = (sum(p.numel() for p in model.conv1.parameters()) +
                        sum(p.numel() for p in model.conv2.parameters()) +
                        sum(p.numel() for p in model.fc2.parameters()) +
                        sum(p.numel() for p in model.fc3.parameters()))
    qnn_params       = sum(p.numel() for p in model.qnn.parameters())
    total            = quanv_params + classical_params + qnn_params

    print(f'\n── {label} Parameter Count ──────────────────')
    print(f'  Quantum front-end (trainable circuit) : {quanv_params:>6}')
    print(f'  Classical backbone (Conv+FC)          : {classical_params:>6}')
    print(f'  QNN back-end (RealAmplitudes)         : {qnn_params:>6}')
    print(f'  ─────────────────────────────────────────────')
    print(f'  Total                                 : {total:>6}')
    return {'quanv': quanv_params, 'classical': classical_params,
            'qnn': qnn_params, 'total': total}

# Instantiate one model just to check parameter counts
# (Need LazyLinear to infer input size first — do a dummy forward pass)
print('Building test model for parameter count...')
test_model = M2_Net('circuit4')
# Trigger LazyLinear initialization with a dummy forward pass
dummy = torch.zeros(1, 28, 28, 1)
with torch.no_grad():
    # Do just the classical part to init LazyLinear without running quantum layers
    feat = torch.zeros(1, 4, 14, 14)  # expected shape after quanv
    out  = F.relu(test_model.conv1(feat))
    out  = test_model.pool1(out)
    out  = F.relu(test_model.conv2(out))
    out  = test_model.pool2(out)
    out  = torch.flatten(out, 1)
    _    = test_model.fc2(out)  # this initializes the LazyLinear
param_counts = count_parameters(test_model, 'M2 Circuit4')
del test_model

## 6. Training Infrastructure

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

def make_pytorch_loaders(x_np, y_np, x_v_np, y_v_np, batch_size=1):
    """Convert numpy arrays to PyTorch DataLoaders."""
    x_t = torch.tensor(x_np, dtype=torch.float32)
    y_t = torch.tensor(y_np, dtype=torch.long)
    x_v = torch.tensor(x_v_np, dtype=torch.float32)
    y_v = torch.tensor(y_v_np, dtype=torch.long)

    tr_loader = DataLoader(TensorDataset(x_t, y_t), batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(x_v, y_v), batch_size=batch_size, shuffle=False)
    return tr_loader, va_loader


def train_m2_model(circuit_name, x_train, y_train, x_val, y_val, cfg):
    """
    Full end-to-end training of M2 with a given Hur et al. circuit as front-end.
    
    The quantum front-end weights ARE trained in this loop — the quanvolutional
    layer is called on every forward pass and its parameters update via
    the parameter-shift rule through PennyLane's TorchLayer.
    
    Returns: (history_dict, final_model, best_val_acc)
    """
    print(f'\n{"="*60}')
    print(f'  Training M2 with {circuit_name.upper()}')
    print(f'{"="*60}')

    model   = M2_Net(circuit_name)
    # Init LazyLinear by running dummy tensor through classical layers
    with torch.no_grad():
        feat = torch.zeros(1, 4, 14, 14)
        out  = F.relu(model.conv1(feat))
        out  = model.pool1(out)
        out  = F.relu(model.conv2(out))
        out  = model.pool2(out)
        out  = torch.flatten(out, 1)
        _    = model.fc2(out)

    count_parameters(model, circuit_name)

    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'])
    loss_fn   = nn.CrossEntropyLoss()

    tr_loader, va_loader = make_pytorch_loaders(
        x_train, y_train, x_val, y_val, batch_size=cfg['batch_size']
    )

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc  = 0.0
    patience_cnt  = 0
    best_state    = None
    t_start       = time.time()

    for epoch in range(1, cfg['epochs'] + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0

        for xb, yb in tr_loader:
            optimizer.zero_grad()
            out  = model(xb)          # full forward pass incl. quantum front-end
            loss = loss_fn(out, yb)
            loss.backward()           # param-shift gradients from TorchLayer
            optimizer.step()
            tr_loss    += loss.item() * len(xb)
            tr_correct += (out.argmax(1) == yb).sum().item()
            tr_total   += len(xb)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        va_loss, va_correct, va_total = 0.0, 0, 0

        with torch.no_grad():
            for xb, yb in va_loader:
                out  = model(xb)
                loss = loss_fn(out, yb)
                va_loss    += loss.item() * len(xb)
                va_correct += (out.argmax(1) == yb).sum().item()
                va_total   += len(xb)

        tr_acc = tr_correct / tr_total
        va_acc = va_correct / va_total
        history['train_loss'].append(tr_loss / tr_total)
        history['val_loss'].append(va_loss / va_total)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        elapsed = time.time() - t_start
        if epoch % 5 == 0 or epoch == 1:
            print(f'  Ep {epoch:3d} | tr_loss {tr_loss/tr_total:.4f} '
                  f'tr_acc {tr_acc:.4f} | val_loss {va_loss/va_total:.4f} '
                  f'val_acc {va_acc:.4f} | {elapsed:.0f}s')

        # Barren plateau check — warn if weights not changing after first 5 epochs
        if epoch == 5:
            quanv_weights = model.quanv.torch_layer.weights.detach().cpu().numpy()
            if np.allclose(quanv_weights, quanv_weights[0], atol=1e-4):
                print(f'  ⚠ WARNING: front-end weights appear flat at epoch 5 — '
                      f'possible barren plateau. weights={np.round(quanv_weights, 4)}')
            else:
                print(f'  ✓ Front-end weights updating: {np.round(quanv_weights, 3)}')

        # Early stopping
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            patience_cnt = 0
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            save_path    = os.path.join(cfg['results_dir'], f'm2_{circuit_name}_best.pt')
            torch.save(best_state, save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= cfg['patience']:
                print(f'  Early stop at epoch {epoch} (best val: {best_val_acc:.4f})')
                break

    # Report final front-end weights
    final_weights = model.quanv.torch_layer.weights.detach().cpu().numpy()
    print(f'\n  Final front-end weights: {np.round(final_weights, 4)}')
    print(f'  Best val acc: {best_val_acc:.4f}')

    model.load_state_dict(best_state)
    return history, model, best_val_acc


print('Training infrastructure ready.')

## 7. Smoke Test — Verify Gradient Flow Before Full Runs

**Run this section first before the full circuit selection experiment.**  
It verifies that:
1. The front-end weights actually update (not frozen / barren plateau)
2. Loss decreases over 10 epochs on 200 samples
3. All three circuits can be instantiated and run without errors

If front-end weights don't change after 5 epochs, a barren plateau is likely — check initialisation scale.

In [ ]:
if SMOKE_TEST:
    print('='*60)
    print('  SMOKE TEST — Gradient flow verification')
    print('  50 samples/class, 10 epochs, Circuit 4 only')
    print('='*60)

    smoke_cfg = {
        **CFG,
        'epochs'  : 10,
        'patience': 15,   # no early stopping during smoke test
    }

    smoke_hist, smoke_model, smoke_best = train_m2_model(
        'circuit4', x_train, y_train, x_val, y_val, smoke_cfg
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(smoke_hist['train_loss'], label='train')
    ax1.plot(smoke_hist['val_loss'],   label='val')
    ax1.set_title('Smoke test — loss (circuit4)')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(smoke_hist['train_acc'], label='train')
    ax2.plot(smoke_hist['val_acc'],   label='val')
    ax2.set_title('Smoke test — accuracy (circuit4)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylim(0, 1)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(CFG['results_dir'], 'smoke_test_circuit4.png'), dpi=150)
    plt.show()

    print('\nSmoke test complete.')
    print('If loss decreased and weights updated → proceed to full circuit selection run.')
    print('If weights flat → barren plateau — reduce learning rate or reinitialise.')
else:
    print('SMOKE_TEST = False → skipping smoke test, proceeding to full run.')
    print('Set SMOKE_TEST = True in the config cell to run gradient verification first.')

## 8. Full Circuit Selection Experiment

Trains all three circuits on MNIST 3v5 with identical hyperparameters.  
Each model trains from scratch with freshly initialised front-end weights.

**Expected runtime on Gadi gpuvolta (V100):**  
- Per epoch: ~2–5 min per circuit (quantum layers on CPU, batch_size=1)
- 30 epochs × 3 circuits: ~3–8 hours total
- Submit as a PBS job for long runs

In [ ]:
if not SMOKE_TEST:
    all_histories  = {}
    all_models     = {}
    all_best_accs  = {}

    for circuit_name in ['circuit2', 'circuit4', 'circuit6']:
        torch.manual_seed(CFG['seed'])  # Reset seed for each circuit — fair comparison
        hist, mdl, best_acc = train_m2_model(
            circuit_name, x_train, y_train, x_val, y_val, CFG
        )
        all_histories[circuit_name] = hist
        all_models[circuit_name]    = mdl
        all_best_accs[circuit_name] = best_acc

    # Save histories
    history_path = os.path.join(CFG['results_dir'], 'circuit_selection_histories.json')
    with open(history_path, 'w') as f:
        json.dump(all_histories, f, indent=2)
    print(f'\nHistories saved to {history_path}')
else:
    print('SMOKE_TEST = True → skipping full circuit selection run.')

## 9. Evaluation on Test Set

In [ ]:
if not SMOKE_TEST:
    x_test_t = torch.tensor(x_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)
    test_dataset = TensorDataset(x_test_t, y_test_t)
    test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

    test_results = {}

    for circuit_name, model in all_models.items():
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                out = model(xb)
                correct += (out.argmax(1) == yb).sum().item()
                total   += len(xb)
        test_acc = correct / total
        test_results[circuit_name] = test_acc
        print(f'{circuit_name}: test acc = {test_acc:.4f}')

    # Save results
    results_summary = {
        'best_val_acc'  : all_best_accs,
        'test_acc'      : test_results,
    }
    with open(os.path.join(CFG['results_dir'], 'circuit_selection_results.json'), 'w') as f:
        json.dump(results_summary, f, indent=2)
    print('\nResults saved.')
else:
    print('SMOKE_TEST = True → skipping test evaluation.')

## 10. Results Visualisation

In [ ]:
if not SMOKE_TEST:
    COLORS = {'circuit2': '#1f77b4', 'circuit4': '#ff7f0e', 'circuit6': '#2ca02c'}
    LABELS = {
        'circuit2': 'Circuit 2 (ablation: Ry-only, 4p, 2 CNOT)',
        'circuit4': 'Circuit 4 (primary: Ry+Rz, 6p, 2 CNOT)',
        'circuit6': 'Circuit 6 (SO(4), 6p, 3 CNOT)',
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Training curves ──────────────────────────────────────────────────────
    for cname in ['circuit2', 'circuit4', 'circuit6']:
        hist = all_histories[cname]
        n    = len(hist['val_acc'])
        axes[0].plot(range(1, n+1), hist['train_acc'],
                     color=COLORS[cname], linestyle='--', alpha=0.5)
        axes[0].plot(range(1, n+1), hist['val_acc'],
                     color=COLORS[cname], label=LABELS[cname])

    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Training curves — M2 circuit selection\n(solid=val, dashed=train)')
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0.4, 1.0)
    axes[0].axhline(0.984, color='grey', linestyle=':', alpha=0.5, label='M1 baseline (98.4%)')
    axes[0].grid(True, alpha=0.3)

    # ── Final accuracy bar chart ─────────────────────────────────────────────
    circuits    = ['circuit2', 'circuit4', 'circuit6']
    val_accs    = [all_best_accs[c] for c in circuits]
    test_accs   = [test_results[c]  for c in circuits]
    short_names = ['C2 (4p, 2 CNOT)', 'C4 (6p, 2 CNOT)', 'C6 (6p, 3 CNOT)']

    x_pos  = np.arange(len(circuits))
    width  = 0.35
    bars1  = axes[1].bar(x_pos - width/2, val_accs,  width,
                          color=[COLORS[c] for c in circuits], alpha=0.7, label='Val acc')
    bars2  = axes[1].bar(x_pos + width/2, test_accs, width,
                          color=[COLORS[c] for c in circuits], alpha=1.0, label='Test acc')

    axes[1].axhline(0.984, color='grey', linestyle=':', alpha=0.7, label='M1 baseline (98.4%)')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(short_names, fontsize=9)
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Circuit selection — final accuracies')
    axes[1].legend()
    axes[1].set_ylim(0.5, 1.0)

    # Annotate bars
    for bar, acc in zip(bars1, val_accs):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{acc:.3f}', ha='center', va='bottom', fontsize=8)
    for bar, acc in zip(bars2, test_accs):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{acc:.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(CFG['results_dir'], 'circuit_selection_results.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved.')

## 11. Decision — Lock the M2 Circuit

In [ ]:
if not SMOKE_TEST:
    print('\n' + '='*60)
    print('  CIRCUIT SELECTION DECISION')
    print('='*60)

    print('\nFinal results:')
    print(f'{"Circuit":<12} {"Val acc":>10} {"Test acc":>10} {"Params":>8} {"CNOTs":>6}')
    print('-' * 52)
    specs_summary = {
        'circuit2': {'params': 4, 'cnots': 2},
        'circuit4': {'params': 6, 'cnots': 2},
        'circuit6': {'params': 6, 'cnots': 3},
    }
    for cname in ['circuit2', 'circuit4', 'circuit6']:
        print(f'{cname:<12} {all_best_accs[cname]:>10.4f} {test_results[cname]:>10.4f}'
              f'{specs_summary[cname]["params"]:>8} {specs_summary[cname]["cnots"]:>6}')

    # Decision rule: highest test acc; tiebreak = fewer CNOTs
    best_test  = max(test_results, key=lambda c: (
        test_results[c], -specs_summary[c]['cnots']
    ))
    second     = sorted(
        test_results.keys(),
        key=lambda c: (test_results[c], -specs_summary[c]['cnots']),
        reverse=True
    )[1]

    print(f'\n  ✓ WINNER: {best_test.upper()}')
    print(f'    Test acc : {test_results[best_test]:.4f}')
    print(f'    Runner-up: {second} ({test_results[second]:.4f})')
    print(f'    M1 baseline: 0.984')
    delta = test_results[best_test] - 0.984
    print(f'    Delta vs M1: {delta:+.4f}')
    if delta > 0:
        print('    → Trainable front-end IMPROVES on M1. Publishable positive result.')
    elif delta > -0.02:
        print('    → Trainable front-end roughly matches M1. Publishable null result.')
    else:
        print('    → Trainable front-end underperforms M1. Investigate barren plateau.')

    # Save decision
    decision = {
        'winner'          : best_test,
        'winner_test_acc' : test_results[best_test],
        'm1_baseline'     : 0.984,
        'delta_vs_m1'     : delta,
        'all_results'     : {c: {'val': all_best_accs[c], 'test': test_results[c]}
                             for c in test_results},
    }
    with open(os.path.join(CFG['results_dir'], 'circuit_selection_decision.json'), 'w') as f:
        json.dump(decision, f, indent=2)
    print(f'\n  Decision saved to {CFG["results_dir"]}/circuit_selection_decision.json')
    print('  This circuit is now the locked M2 front-end for all subsequent experiments.')
else:
    print('SMOKE_TEST = True — run full experiment to generate circuit selection decision.')

## 12. Front-End Weight Analysis

Examine the trained weights to check for barren plateau indicators.
A well-trained circuit should have weights spread across [−π, π], not clustered near zero.

In [ ]:
if not SMOKE_TEST:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for idx, (cname, model) in enumerate(all_models.items()):
        weights = model.quanv.torch_layer.weights.detach().cpu().numpy()
        axes[idx].bar(range(len(weights)), weights, color=COLORS[cname])
        axes[idx].axhline(0, color='grey', linewidth=0.5)
        axes[idx].set_title(f'{cname} — trained weights')
        axes[idx].set_xlabel('Parameter index')
        axes[idx].set_ylabel('Weight value (radians)')
        axes[idx].set_ylim(-np.pi, np.pi)
        axes[idx].axhline(np.pi,  color='red', linestyle='--', alpha=0.3, linewidth=0.5)
        axes[idx].axhline(-np.pi, color='red', linestyle='--', alpha=0.3, linewidth=0.5)

        # Barren plateau heuristic: if weights all near zero → potential problem
        weight_spread = np.std(weights)
        bp_flag = '⚠ possible barren plateau' if weight_spread < 0.1 else '✓ weights active'
        axes[idx].set_xlabel(f'Weight std: {weight_spread:.3f} — {bp_flag}')

    plt.tight_layout()
    plt.savefig(os.path.join(CFG['results_dir'], 'trained_weights.png'), dpi=150)
    plt.show()

## 13. PBS Job Script

Convenience cell — prints the PBS submission command for Gadi.  
The full run is best submitted as a PBS job rather than run interactively in ARE.

In [ ]:
pbs_script = '''
#!/bin/bash
#PBS -N m2_circuit_selection
#PBS -q gpuvolta
#PBS -l ncpus=12
#PBS -l ngpus=1
#PBS -l mem=48GB
#PBS -l walltime=12:00:00
#PBS -l jobfs=10GB
#PBS -l storage=gdata/wa66+scratch/wa66
#PBS -P wa66
#PBS -o /g/data/wa66/yh2481/logs/circuit_selection_out.txt
#PBS -e /g/data/wa66/yh2481/logs/circuit_selection_err.txt

cd /g/data/wa66/yh2481
source /g/data/wa66/yh2481/miniconda3/etc/profile.d/conda.sh
conda activate qcqcnn

# PYTHONPATH must be cleared to prevent system Python leaking into conda env
PYTHONPATH="" jupyter nbconvert --to notebook \\
    --execute m2_circuit_selection.ipynb \\
    --output m2_circuit_selection_executed.ipynb \\
    --ExecutePreprocessor.timeout=43200
'''

print(pbs_script)
print('Save this as run_circuit_selection.pbs and submit with:')
print('  qsub run_circuit_selection.pbs')
print('\nCheck queue status with:')
print('  qstat -u $USER')
print('\nResults written to: results_circuit_selection/')
print('  - circuit_selection_results.png')
print('  - circuit_selection_decision.json  ← the locked M2 circuit')
print('  - trained_weights.png')
print('  - m2_circuit2_best.pt')
print('  - m2_circuit4_best.pt')
print('  - m2_circuit6_best.pt')